# SofaScore Player Average Ratings by Team

This notebook calculates average SofaScore ratings for every player meeting the existing appearance rule in the recent **overall** and **Bundesliga** match windows. The categories are evaluated independently: ratings are never mixed between them.

Enter the current Bundesliga matchday when prompted. On matchdays 1–3, a player is included with valid ratings in at least two of the selected five matches, or with a valid rating in the immediately latest match. From matchday 4, the normal rule applies: at least three valid ratings, or valid ratings in both of the two most-recent available matches. Missing or non-numeric ratings are ignored. Unlike the high-rated-players notebook, no minimum average-rating threshold is applied.

Outputs are timestamped JSON and CSV files in `outputs/sofascore/player_average_ratings`. JSON retains each player's per-match rating trail; CSV has one row per player and category.


In [1]:
from __future__ import annotations

import csv
import json
import math
import re
import sys
import warnings
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Any


def locate_project_root() -> Path:
    starts = []
    notebook_path = globals().get('__vsc_ipynb_file__')
    if isinstance(notebook_path, str) and notebook_path.strip():
        starts.append(Path(notebook_path).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project_paths.py. Start Jupyter from the project root.')


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    SOFASCORE_PLAYER_AVERAGE_RATINGS_DIR,
    SOFASCORE_TEAM_FORM_DIR,
    ensure_directory,
)

try:
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError('Install undetected-chromedriver, selenium, and beautifulsoup4 in this Jupyter kernel, then restart it.') from exc

CHROME_MAJOR_VERSION = 150
HEADLESS = False
PAGE_LOAD_TIMEOUT_SECONDS = 30
WAIT_TIMEOUT_SECONDS = 20
EARLY_SEASON_MAX_MATCHDAY = 3
EARLY_SEASON_MINIMUM_RATED_MATCHES = 2
MINIMUM_RATED_MATCHES = 3
MINIMUM_BUNDESLIGA_MATCHDAY = 1
MAXIMUM_BUNDESLIGA_MATCHDAY = 34
LINEUPS_URL_TEMPLATE = 'https://www.sofascore.com/api/v1/event/{match_id}/lineups'
TEAM_FORM_FILENAME_RE = re.compile(r'^team_form_(?P<date>\d{4}-\d{2}-\d{2})_(?P<time>\d{2}-\d{2}-\d{2})(?:_(?P<microseconds>\d{1,6}))?(?P<offset>[+-]\d{4})?\.json$')

for setting_name, setting_value in {
    'EARLY_SEASON_MAX_MATCHDAY': EARLY_SEASON_MAX_MATCHDAY,
    'EARLY_SEASON_MINIMUM_RATED_MATCHES': EARLY_SEASON_MINIMUM_RATED_MATCHES,
    'MINIMUM_RATED_MATCHES': MINIMUM_RATED_MATCHES,
    'MINIMUM_BUNDESLIGA_MATCHDAY': MINIMUM_BUNDESLIGA_MATCHDAY,
    'MAXIMUM_BUNDESLIGA_MATCHDAY': MAXIMUM_BUNDESLIGA_MATCHDAY,
}.items():
    if not isinstance(setting_value, int) or isinstance(setting_value, bool) or setting_value < 1:
        raise ValueError(f'{setting_name} must be a positive integer.')
if MINIMUM_BUNDESLIGA_MATCHDAY > EARLY_SEASON_MAX_MATCHDAY or EARLY_SEASON_MAX_MATCHDAY >= MAXIMUM_BUNDESLIGA_MATCHDAY:
    raise ValueError('The early-season matchday range must fall within the Bundesliga season.')


def prompt_bundesliga_matchday() -> int:
    raw_matchday = input(f'Current Bundesliga matchday ({MINIMUM_BUNDESLIGA_MATCHDAY}-{MAXIMUM_BUNDESLIGA_MATCHDAY}): ').strip()
    if not raw_matchday:
        raise ValueError('Bundesliga matchday is required.')
    try:
        matchday = int(raw_matchday)
    except ValueError as exc:
        raise ValueError('Bundesliga matchday must be a whole number.') from exc
    if not MINIMUM_BUNDESLIGA_MATCHDAY <= matchday <= MAXIMUM_BUNDESLIGA_MATCHDAY:
        raise ValueError(f'Bundesliga matchday must be between {MINIMUM_BUNDESLIGA_MATCHDAY} and {MAXIMUM_BUNDESLIGA_MATCHDAY}.')
    return matchday


BUNDESLIGA_MATCHDAY = prompt_bundesliga_matchday()


Current Bundesliga matchday (1-34):  1


## Load the latest team-form snapshot

The latest valid filename timestamp determines the input, rather than alphabetical order or modification time.


In [2]:
class TeamFormInputError(ValueError):
    """Raised when the selected team-form snapshot is unusable."""


def is_finite_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value))


def parse_iso_datetime(value: Any) -> datetime:
    if not isinstance(value, str) or not value.strip():
        raise ValueError('date must be a non-empty ISO datetime string')
    normalized = value.strip().replace('Z', '+00:00') if value.strip().endswith('Z') else value.strip()
    parsed = datetime.fromisoformat(normalized)
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=datetime.now().astimezone().tzinfo)
    return parsed.astimezone(timezone.utc)


def match_chronology_value(match: dict[str, Any]) -> datetime:
    if is_finite_number(match.get('timestamp')):
        try:
            return datetime.fromtimestamp(float(match['timestamp']), tz=timezone.utc)
        except (OverflowError, OSError, ValueError):
            pass
    try:
        return parse_iso_datetime(match.get('date'))
    except (TypeError, ValueError) as exc:
        raise TeamFormInputError(f"Match {match.get('match_id')!r} has no usable timestamp or date.") from exc


def parse_team_form_filename_timestamp(path: Path) -> datetime:
    match = TEAM_FORM_FILENAME_RE.fullmatch(path.name)
    if match is None:
        raise ValueError(f'Unsupported team-form filename: {path.name}')
    parsed = datetime.strptime(f"{match.group('date')}_{match.group('time')}", '%Y-%m-%d_%H-%M-%S')
    if match.group('microseconds'):
        parsed = parsed.replace(microsecond=int(match.group('microseconds').ljust(6, '0')))
    offset = match.group('offset')
    tzinfo = datetime.strptime(offset, '%z').tzinfo if offset else datetime.now().astimezone().tzinfo
    return parsed.replace(tzinfo=tzinfo).astimezone(timezone.utc)


def select_latest_team_form_file(directory: Path) -> Path:
    parsed, malformed = [], []
    for path in sorted(directory.glob('team_form_*.json')):
        try:
            parsed.append((parse_team_form_filename_timestamp(path), path))
        except ValueError as exc:
            malformed.append(f'{path.name}: {exc}')
    for message in malformed:
        warnings.warn(f'Ignoring {message}', stacklevel=2)
    if not parsed:
        raise FileNotFoundError(f'No valid team_form_*.json files in {directory}.')
    latest = max(timestamp for timestamp, _ in parsed)
    paths = [path for timestamp, path in parsed if timestamp == latest]
    if len(paths) != 1:
        raise RuntimeError('Multiple team-form files encode the same latest instant.')
    return paths[0]


def validate_match(raw_match: Any, team_id: int, category: str) -> dict[str, Any]:
    if not isinstance(raw_match, dict):
        raise TeamFormInputError(f'Team {team_id} {category} has a non-object match.')
    match = dict(raw_match)
    if not isinstance(match.get('match_id'), int) or isinstance(match['match_id'], bool) or match['match_id'] < 1:
        raise TeamFormInputError(f'Team {team_id} {category} has an invalid match_id.')
    sides = (match.get('home_team_id'), match.get('away_team_id'))
    if any(not isinstance(side, int) or isinstance(side, bool) or side < 1 for side in sides):
        raise TeamFormInputError(f'Match {match["match_id"]} has invalid team IDs.')
    if team_id not in sides:
        raise TeamFormInputError(f'Match {match["match_id"]} does not contain team {team_id}.')
    match_chronology_value(match)
    return match


def load_team_form_snapshot(path: Path) -> dict[int, dict[str, Any]]:
    try:
        raw = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise TeamFormInputError(f'Could not load team-form input {path}: {exc}') from exc
    if not isinstance(raw, dict) or not raw:
        raise TeamFormInputError('The team-form JSON must be a non-empty object.')
    if all(str(key).isdigit() for key in raw):
        raw_teams = raw
    elif len(raw) == 1:
        snapshot_key, raw_teams = next(iter(raw.items()))
        parse_iso_datetime(snapshot_key)
    else:
        raise TeamFormInputError('Expected a team-ID object or one timestamped snapshot object.')
    if not isinstance(raw_teams, dict) or not raw_teams:
        raise TeamFormInputError('The snapshot must contain a non-empty team object.')
    teams = {}
    for key, raw_team in raw_teams.items():
        try:
            team_id = int(str(key))
        except (TypeError, ValueError) as exc:
            raise TeamFormInputError(f'Invalid team key: {key!r}') from exc
        if team_id < 1 or str(team_id) != str(key) or not isinstance(raw_team, dict):
            raise TeamFormInputError(f'Invalid team record: {key!r}')
        team_name = raw_team.get('team')
        if not isinstance(team_name, str) or not team_name.strip():
            raise TeamFormInputError(f'Team {team_id} has no valid name.')
        record = {'team': team_name.strip()}
        for category, field in (('overall', 'overall_matches'), ('bundesliga', 'bundesliga_matches')):
            matches = raw_team.get(field)
            if not isinstance(matches, list):
                raise TeamFormInputError(f'Team {team_id} field {field} must be a list.')
            validated = [validate_match(item, team_id, category) for item in matches]
            if len({item['match_id'] for item in validated}) != len(validated):
                raise TeamFormInputError(f'Team {team_id} {category} contains duplicate match IDs.')
            record[field] = validated
        teams[team_id] = record
    return teams


if not SOFASCORE_TEAM_FORM_DIR.is_dir():
    raise FileNotFoundError(f'Team-form output directory not found: {SOFASCORE_TEAM_FORM_DIR}. Run the recent-form notebook first.')
selected_input_path = select_latest_team_form_file(SOFASCORE_TEAM_FORM_DIR)
teams = load_team_form_snapshot(selected_input_path)
print(f'Using team-form input: {selected_input_path.name}')
print(f'Loaded and validated {len(teams)} team(s).')


Using team-form input: team_form_2026-08-23_00-46-27_454341+0200.json
Loaded and validated 18 team(s).


## Retrieve and cache SofaScore lineups

One browser session and one cache entry per match are reused across both categories.


In [3]:
class SofaScoreLineupError(RuntimeError):
    """Raised when a lineup endpoint cannot return a usable payload."""


def create_browser() -> Any:
    options = uc.ChromeOptions()
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    if HEADLESS:
        options.add_argument('--headless=new')
    try:
        driver = uc.Chrome(options=options, version_main=CHROME_MAJOR_VERSION, use_subprocess=True)
        driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
        print(f'Chrome ready (major version {CHROME_MAJOR_VERSION}, headless={HEADLESS}).')
        return driver
    except Exception as exc:
        raise RuntimeError(f'Could not initialize Chrome: {type(exc).__name__}: {exc}') from exc


def close_browser(driver: Any | None) -> None:
    if driver is None:
        return
    try:
        driver.quit()
        print('Chrome driver closed.')
    except Exception as exc:
        print(f'Chrome shutdown warning: {type(exc).__name__}: {exc}')


def fetch_lineup_payload(driver: Any, match_id: int) -> dict[str, Any]:
    url = LINEUPS_URL_TEMPLATE.format(match_id=match_id)
    try:
        driver.get(url)
        WebDriverWait(driver, WAIT_TIMEOUT_SECONDS).until(EC.presence_of_element_located((By.TAG_NAME, 'pre')))
    except (TimeoutException, WebDriverException) as exc:
        raise SofaScoreLineupError(f'Could not load {url}: {exc}') from exc
    pre_tag = BeautifulSoup(driver.page_source, 'html.parser').find('pre')
    if pre_tag is None or not pre_tag.get_text().strip():
        raise SofaScoreLineupError(f'Rendered response has no JSON body: {url}')
    try:
        payload = json.loads(pre_tag.get_text())
    except json.JSONDecodeError as exc:
        raise SofaScoreLineupError(f'Invalid JSON at {url}: {exc}') from exc
    if not isinstance(payload, dict) or any(not isinstance(payload.get(side), dict) or not isinstance(payload[side].get('players'), list) for side in ('home', 'away')):
        raise SofaScoreLineupError(f'Lineup response has no usable home/away player lists: {url}')
    return payload


def get_cached_lineup(driver: Any, match_id: int, match_cache: dict[int, dict[str, Any]], counters: dict[str, int]) -> dict[str, Any]:
    if match_id in match_cache:
        counters['cache_hits'] += 1
        return match_cache[match_id]
    try:
        result = {'ok': True, 'payload': fetch_lineup_payload(driver, match_id), 'error': None}
        counters['successful_requests'] += 1
    except Exception as exc:
        result = {'ok': False, 'payload': None, 'error': f'{type(exc).__name__}: {exc}'}
        counters['failed_requests'] += 1
    match_cache[match_id] = result
    return result


## Calculate player averages

Only valid numeric SofaScore ratings count toward an average.


In [4]:
def determine_team_side(match: dict[str, Any], team_id: int) -> str:
    if match['home_team_id'] == team_id:
        return 'home'
    if match['away_team_id'] == team_id:
        return 'away'
    raise TeamFormInputError(f"Match {match['match_id']} does not contain team {team_id}.")


def extract_rated_players_for_side(payload: dict[str, Any], side: str, match_id: int) -> list[dict[str, Any]]:
    extracted, seen_player_ids = [], set()
    for entry in payload[side]['players']:
        if not isinstance(entry, dict):
            continue
        player, statistics = entry.get('player'), entry.get('statistics')
        if not isinstance(player, dict) or not isinstance(statistics, dict):
            continue
        player_id, rating = player.get('id'), statistics.get('rating')
        if not isinstance(player_id, int) or isinstance(player_id, bool) or player_id < 1 or not is_finite_number(rating):
            continue
        if player_id in seen_player_ids:
            raise SofaScoreLineupError(f'Match {match_id} has duplicate rated player {player_id} on the {side} side.')
        seen_player_ids.add(player_id)
        name, position = player.get('name'), entry.get('position')
        extracted.append({'player_id': player_id, 'player_name': name.strip() if isinstance(name, str) and name.strip() else None, 'position': position.strip() if isinstance(position, str) and position.strip() else None, 'rating': float(rating), 'match_id': match_id})
    return extracted


def failed_match_context(match_id: int, team_id: int, team_name: str, category: str, error: str) -> dict[str, Any]:
    return {'match_id': match_id, 'team_id': team_id, 'team': team_name, 'category': category, 'error': error}


def eligibility_metadata(matchday: int) -> dict[str, Any]:
    early_season = matchday <= EARLY_SEASON_MAX_MATCHDAY
    return {
        'bundesliga_matchday': matchday,
        'early_season_max_matchday': EARLY_SEASON_MAX_MATCHDAY,
        'minimum_rated_matches': EARLY_SEASON_MINIMUM_RATED_MATCHES if early_season else MINIMUM_RATED_MATCHES,
        'last_match_exception': early_season,
        'last_two_exception': not early_season,
    }


def player_qualifies_for_average(rating_count: int, rated_match_ids: set[int], ordered_match_ids: list[int], matchday: int) -> bool:
    if matchday <= EARLY_SEASON_MAX_MATCHDAY:
        played_latest_match = bool(ordered_match_ids) and ordered_match_ids[-1] in rated_match_ids
        return rating_count >= EARLY_SEASON_MINIMUM_RATED_MATCHES or played_latest_match
    last_two_ids = set(ordered_match_ids[-2:]) if len(ordered_match_ids) >= 2 else set()
    played_last_two = bool(last_two_ids) and last_two_ids.issubset(rated_match_ids)
    return rating_count >= MINIMUM_RATED_MATCHES or played_last_two


def eligibility_rule_description(matchday: int) -> str:
    if matchday <= EARLY_SEASON_MAX_MATCHDAY:
        return f'at least {EARLY_SEASON_MINIMUM_RATED_MATCHES} rated matches or the immediately latest match'
    return f'at least {MINIMUM_RATED_MATCHES} rated matches or both latest matches'


def evaluate_category(driver: Any, team_id: int, team_name: str, category: str, matches: list[dict[str, Any]], match_cache: dict[int, dict[str, Any]], counters: dict[str, int], failed_matches: list[dict[str, Any]]) -> list[dict[str, Any]]:
    ordered = sorted(matches, key=lambda match: (match_chronology_value(match), match['match_id']))
    ordered_match_ids = [match['match_id'] for match in ordered]
    buckets = {}
    for match in ordered:
        cached = get_cached_lineup(driver, match['match_id'], match_cache, counters)
        if not cached['ok']:
            failed_matches.append(failed_match_context(match['match_id'], team_id, team_name, category, str(cached['error'])))
            continue
        try:
            rated_players = extract_rated_players_for_side(cached['payload'], determine_team_side(match, team_id), match['match_id'])
        except Exception as exc:
            failed_matches.append(failed_match_context(match['match_id'], team_id, team_name, category, f'{type(exc).__name__}: {exc}'))
            continue
        for rated in rated_players:
            bucket = buckets.setdefault(rated['player_id'], {'player_name': None, 'position': None, 'ratings': [], 'rated_match_ids': set()})
            bucket['player_name'] = rated['player_name'] or bucket['player_name']
            bucket['position'] = rated['position'] or bucket['position']
            bucket['ratings'].append({'match_id': rated['match_id'], 'rating': rated['rating']})
            bucket['rated_match_ids'].add(rated['match_id'])
    results = []
    for player_id, bucket in buckets.items():
        ratings = bucket['ratings']
        rating_count = len(ratings)
        if not player_qualifies_for_average(rating_count, bucket['rated_match_ids'], ordered_match_ids, BUNDESLIGA_MATCHDAY):
            continue
        average = sum((Decimal(str(record['rating'])) for record in ratings), Decimal('0')) / Decimal(rating_count)
        results.append((average, {'player_id': player_id, 'player_name': bucket['player_name'], 'position': bucket['position'], 'rating_count': rating_count, 'average_rating': float(average.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)), 'ratings': ratings}))
    results.sort(key=lambda item: (-item[0], (item[1]['player_name'] or '').casefold(), item[1]['player_id']))
    return [result for _, result in results]


def process_all_teams(driver: Any, teams: dict[int, dict[str, Any]], match_cache: dict[int, dict[str, Any]], counters: dict[str, int]) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    failures, results = [], {}
    for number, (team_id, record) in enumerate(teams.items(), start=1):
        print(f"[{number}/{len(teams)}] {record['team']} (team_id={team_id})")
        results[str(team_id)] = {'team': record['team'], 'overall': {'players': evaluate_category(driver, team_id, record['team'], 'overall', record['overall_matches'], match_cache, counters, failures)}, 'bundesliga': {'players': evaluate_category(driver, team_id, record['team'], 'bundesliga', record['bundesliga_matches'], match_cache, counters, failures)}}
    return results, failures


## Run and export

The `finally` block closes Chrome even if a request or export fails.


In [5]:
CSV_COLUMNS = ['team_id', 'team', 'category', 'player_id', 'player_name', 'position', 'rating_count', 'average_rating']


def export_results(source_path: Path, generated_datetime: datetime, team_results: dict[str, Any], failed_matches: list[dict[str, Any]]) -> tuple[Path, Path]:
    timestamp = generated_datetime.strftime('%Y-%m-%d_%H-%M-%S_%f%z')
    output_dir = ensure_directory(SOFASCORE_PLAYER_AVERAGE_RATINGS_DIR)
    json_path = output_dir / f'team_player_average_ratings_{timestamp}.json'
    csv_path = output_dir / f'team_player_average_ratings_{timestamp}.csv'
    document = {'generated_at': generated_datetime.isoformat(timespec='seconds'), 'source_file': source_path.name, 'eligibility': eligibility_metadata(BUNDESLIGA_MATCHDAY), 'teams': team_results, 'failed_matches': failed_matches}
    try:
        json_path.write_text(json.dumps(document, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        with csv_path.open('w', encoding='utf-8-sig', newline='') as handle:
            writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS)
            writer.writeheader()
            for team_id, team_result in team_results.items():
                for category in ('overall', 'bundesliga'):
                    for player in team_result[category]['players']:
                        writer.writerow({'team_id': team_id, 'team': team_result['team'], 'category': category, 'player_id': player['player_id'], 'player_name': player['player_name'], 'position': player['position'], 'rating_count': player['rating_count'], 'average_rating': f"{player['average_rating']:.2f}"})
    except OSError as exc:
        raise OSError(f'Could not write output files: {exc}') from exc
    return json_path, csv_path


def unique_input_match_ids(teams: dict[int, dict[str, Any]]) -> set[int]:
    return {match['match_id'] for record in teams.values() for field in ('overall_matches', 'bundesliga_matches') for match in record[field]}


def print_summary(team_results: dict[str, Any], counters: dict[str, int], json_path: Path, csv_path: Path) -> None:
    overall_count = sum(len(team['overall']['players']) for team in team_results.values())
    bundesliga_count = sum(len(team['bundesliga']['players']) for team in team_results.values())
    print('\nProcessing summary')
    print('------------------')
    print(f'Selected input file: {selected_input_path.name}')
    print(f'Bundesliga matchday: {BUNDESLIGA_MATCHDAY}')
    print(f'Eligibility rule (overall and Bundesliga): {eligibility_rule_description(BUNDESLIGA_MATCHDAY)}')
    print(f'Teams processed: {len(team_results)} / {len(teams)}')
    print(f'Unique match IDs encountered: {len(unique_input_match_ids(teams))}')
    print(f"Successful SofaScore requests: {counters['successful_requests']}")
    print(f"Failed SofaScore requests: {counters['failed_requests']}")
    print(f"Cache hits / reused matches: {counters['cache_hits']}")
    print(f'Included overall players: {overall_count}')
    print(f'Included Bundesliga players: {bundesliga_count}')
    print(f'JSON output: {json_path}')
    print(f'CSV output: {csv_path}')


driver = None
match_cache: dict[int, dict[str, Any]] = {}
request_counters = {'successful_requests': 0, 'failed_requests': 0, 'cache_hits': 0}
try:
    driver = create_browser()
    final_team_results, failed_matches = process_all_teams(driver, teams, match_cache, request_counters)
    json_output_path, csv_output_path = export_results(selected_input_path, datetime.now().astimezone(), final_team_results, failed_matches)
    print_summary(final_team_results, request_counters, json_output_path, csv_output_path)
finally:
    close_browser(driver)


Chrome ready (major version 150, headless=False).
[1/18] FC Bayern München (team_id=2672)
[2/18] VfB Stuttgart (team_id=2677)
[3/18] 1. FC Köln (team_id=2671)
[4/18] TSG Hoffenheim (team_id=2569)
[5/18] 1. FC Union Berlin (team_id=2547)
[6/18] Eintracht Frankfurt (team_id=2674)
[7/18] 1. FSV Mainz 05 (team_id=2556)
[8/18] SC Paderborn 07 (team_id=2561)
[9/18] RB Leipzig (team_id=36360)
[10/18] Borussia M'gladbach (team_id=2527)
[11/18] SV 07 Elversberg (team_id=2598)
[12/18] Bayer 04 Leverkusen (team_id=2681)
[13/18] Borussia Dortmund (team_id=2673)
[14/18] Hamburger SV (team_id=2676)
[15/18] SC Freiburg (team_id=2538)
[16/18] SV Werder Bremen (team_id=2534)
[17/18] FC Augsburg (team_id=2600)
[18/18] FC Schalke 04 (team_id=2530)

Processing summary
------------------
Selected input file: team_form_2026-08-23_00-46-27_454341+0200.json
Bundesliga matchday: 1
Eligibility rule (overall and Bundesliga): at least 2 rated matches or the immediately latest match
Teams processed: 18 / 18
Unique